[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/11.%20The%20Dynamic%20Berth%20Allocation%20Problem/P11-Tier-7_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 11. The Dynamic Berth Allocation Problem
## Tier 7 — Human–AI Symbiotic Partnership

### Goal
This notebook demonstrates a rule-based Human–AI partnership for sequential berth assignment. An explainable AI component ranks feasible berth–time assignments, a human-operator model selects an appropriate decision mode, and a partnership manager records the resulting schedule and descriptive metrics.

### Inputs and outputs
The input is a fixed teaching instance of nine vessels and three berths. Vessel processing times are expressed in hours; vessel lengths and berth capacities are expressed in metres. The simulation processes vessels in their listed order. It produces one assignment per vessel, a decision log, summary metrics, and visualizations.

### Scope and modeling assumptions
- This is a transparent, illustrative simulation, not an empirical port study, a trained AI system, or a mathematical optimization solver.
- The AI score weights and human decision rules are stipulated teaching rules. They are not learned from data or calibrated against operational outcomes.
- Each berth can serve at most one vessel at a time. Service intervals are half-open: a vessel occupying \([s_i, c_i)\) may be followed by another vessel starting at \(c_i\).
- A vessel may be assigned only to a berth whose length capacity is sufficient. The vessel's desired departure time is contextual information, not a hard deadline.
- Vessels are considered in input order; the simulation does not reorder arrivals or revisit earlier assignments. The resulting schedule is feasible under the stated constraints but is not claimed to be globally optimal.
- Decision duration measures the computer's elapsed execution time for the decision routine. It does not measure a human operator's real-world response time.
- Trust is initialized as a model parameter. Since the simulation does not observe realized service quality, it does not update trust or establish operator satisfaction. Acceptance and satisfaction-related quantities are proxies, not survey findings.

## 1. Partnership model and decision logic

### 1.1 Explainable AI assignment score
For a vessel \(v\), berth \(b\), and feasible start time \(s\), the AI computes a weighted feature score and subtracts a waiting penalty:

\[
S(v,b,s) = \sum_{k \in K} w_k x_k(v,b,s) - \lambda\,W(v,s),
\qquad
W(v,s) = \max(0, s-a_v),
\qquad \lambda = 0.05.
\]

Here, \(K\) is the set of seven features defined in `ExplainableAISystem.feature_weights`; \(w_k\) is a feature weight; and \(x_k\) is the corresponding normalized feature value multiplied by that weight. The feature values represent vessel priority, customer importance, berth productivity, schedule flexibility, historical performance, berth preference, and processing-time efficiency. The waiting time \(W\) is measured in hours, so the score is a teaching heuristic rather than a physical cost with a monetary unit.

For each berth that can accommodate the vessel, the earliest non-conflicting start time at or after arrival is evaluated. The candidate with the greatest score is recommended. The alternatives are the next-highest-scoring feasible candidates. If the best and second-best scores are close, the recommendation receives a lower confidence score; confidence is a rule-based margin indicator, not a statistically calibrated probability.

### 1.2 Situation complexity and decision mode
The operator model combines six bounded factors into a complexity score:

\[
C(v,d) = \min\!\left(1,\sum_{j \in J} q_j z_j(v,d)\right),
\]

where \(J\) contains VIP status, high customer importance, low AI confidence, a tight desired-departure window, low schedule flexibility, and weaker historical performance. The weights \(q_j\) sum to one. Their exact definitions and thresholds are implemented in `HumanOperator.assess_situation_complexity`.

A VIP vessel or a case with \(C>0.7\) is human-led. Otherwise, a recommendation with confidence above 0.85 and complexity below 0.3 is AI-led; remaining cases are collaborative. These thresholds are explicit decision rules, not learned boundaries.

### 1.3 Sequential scheduling and feasibility
A schedule entry for vessel \(v\) on berth \(b\) occupies \([s_v, s_v+p_{vb})\), where \(p_{vb}\) is the berth-specific processing time. A candidate is feasible when it starts no earlier than the vessel's arrival, fits the berth's length capacity, and does not overlap another entry on that berth. The function `earliest_feasible_start` searches forward through existing reservations; the partnership manager checks the final human–AI decision before adding it to the schedule.

This is a sequential decision process, not a global search over all vessel assignments. For each vessel, the AI enumerates the available berths and ranks their earliest feasible assignments; the operator then applies the mode-specific rule. The number of berth candidates evaluated for one vessel is linear in the number of berths, while checking conflicts scans the current schedule. The overall work therefore depends on both the number of vessels and the schedule size, and the method does not guarantee a globally minimum-cost schedule.

In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Fixed seeds keep any future stochastic extensions reproducible.
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully for the Human–AI partnership simulation.")

In [ ]:
class DecisionMode(Enum):
    """Available ways to divide decision authority."""

    AI_LED = "AI-Led"
    HUMAN_LED = "Human-Led"
    COLLABORATIVE = "Collaborative"


@dataclass
class Vessel:
    """Vessel attributes used by the assignment and decision rules."""

    id: int
    arrival_time: int
    desired_departure: int
    length: int
    teu_capacity: int
    processing_times: Dict[int, int]
    priority: str = "Normal"
    customer_importance: float = 1.0
    schedule_flexibility: float = 1.0
    historical_performance: float = 0.95


@dataclass
class Berth:
    """Berth capacity, productivity, and priority preferences."""

    id: int
    capacity: int
    crane_count: int
    productivity: float
    location_preference: Dict[str, float] = field(default_factory=dict)
    current_utilization: float = 0.0


@dataclass
class AIDecision:
    """AI recommendation with its score explanation and alternatives."""

    vessel_id: int
    recommended_berth: int
    recommended_start_time: int
    confidence_score: float
    feature_importance: Dict[str, float]
    alternative_solutions: List[Dict]
    reasoning: str
    processing_time_ms: float


@dataclass
class HumanDecision:
    """Final assignment and decision context for one vessel."""

    vessel_id: int
    final_berth: int
    final_start_time: int
    decision_mode: DecisionMode
    reasoning: str
    confidence_level: float
    time_to_decide_seconds: float
    ai_agreement_score: float


@dataclass
class PartnershipMetrics:
    """Descriptive metrics available from this simulation."""

    total_decisions: int = 0
    ai_led_decisions: int = 0
    human_led_decisions: int = 0
    collaborative_decisions: int = 0
    ai_acceptance_rate: float = 0.0
    average_decision_time_seconds: float = 0.0
    trust_score: float = 0.0
    operator_satisfaction: float = 0.0
    total_service_time_hours: float = 0.0
    performance_improvement_vs_ai: Optional[float] = None
    performance_improvement_vs_human: Optional[float] = None


print("Partnership data structures are ready.")

In [ ]:
def create_partnership_problem_instance() -> Tuple[List[Vessel], List[Berth]]:
    """Build the fixed vessel and berth instance used throughout the notebook."""

    vessels = [
        Vessel(1, 0, 8, 280, 12000, {1: 4, 2: 5, 3: 6}, "VIP", 5.0, 0.2, 0.98),
        Vessel(2, 1, 10, 320, 15000, {1: 5, 2: 6, 3: 7}, "VIP", 4.8, 0.3, 0.97),
        Vessel(3, 2, 12, 250, 8000, {1: 3, 2: 4, 3: 5}, "High", 3.5, 0.6, 0.94),
        Vessel(4, 3, 14, 300, 10000, {1: 4, 2: 5, 3: 6}, "High", 3.2, 0.7, 0.92),
        Vessel(5, 4, 16, 200, 5000, {1: 2, 2: 3, 3: 4}, "Normal", 2.0, 0.8, 0.90),
        Vessel(6, 5, 18, 220, 6000, {1: 3, 2: 3, 3: 4}, "Normal", 2.1, 0.9, 0.88),
        Vessel(7, 6, 20, 180, 4000, {1: 2, 2: 2, 3: 3}, "Normal", 1.9, 1.0, 0.91),
        Vessel(8, 7, 24, 150, 3000, {1: 1, 2: 2, 3: 2}, "Low", 1.0, 1.0, 0.85),
        Vessel(9, 8, 26, 170, 3500, {1: 2, 2: 2, 3: 3}, "Low", 1.2, 1.0, 0.87),
    ]

    berths = [
        Berth(1, 350, 4, 120, {"VIP": 1.0, "High": 0.9, "Normal": 0.8, "Low": 0.6}),
        Berth(2, 400, 5, 150, {"VIP": 0.9, "High": 1.0, "Normal": 0.9, "Low": 0.8}),
        Berth(3, 450, 6, 180, {"VIP": 0.7, "High": 0.8, "Normal": 1.0, "Low": 1.0}),
    ]
    return vessels, berths


vessels, berths = create_partnership_problem_instance()

print("Human–AI partnership problem instance")
print("\nVessels:")
for vessel in vessels:
    print(
        f"  Vessel {vessel.id}: {vessel.priority} priority, "
        f"{vessel.teu_capacity} TEU, length {vessel.length} m, "
        f"importance {vessel.customer_importance:.1f}, "
        f"flexibility {vessel.schedule_flexibility:.1f}"
    )

print("\nBerths:")
for berth in berths:
    print(
        f"  Berth {berth.id}: {berth.capacity} m capacity, "
        f"{berth.crane_count} cranes, productivity {berth.productivity} TEU/hour"
    )

## 2. Explainable AI recommendations

The AI evaluates each berth that can physically accommodate the vessel. For each one, it finds the earliest feasible start time, calculates the seven weighted feature contributions, and applies the waiting penalty. The returned explanation exposes those contributions so that a reader can inspect how the ranking was formed. The recommendation is a heuristic ranking, not a claim that the assignment minimizes total port cost.

In [ ]:
def earliest_feasible_start(
    vessel: Vessel,
    berth: Berth,
    schedule: List[Dict],
) -> Optional[int]:
    """Return the earliest arrival-or-later start that avoids berth conflicts."""

    if vessel.length > berth.capacity:
        return None

    duration = vessel.processing_times[berth.id]
    candidate_start = vessel.arrival_time
    berth_entries = sorted(
        (entry for entry in schedule if entry["berth_id"] == berth.id),
        key=lambda entry: entry["start_time"],
    )

    # Moving the candidate to a conflicting reservation's end preserves
    # half-open intervals and finds the first available gap in schedule order.
    for entry in berth_entries:
        candidate_end = candidate_start + duration
        if candidate_end <= entry["start_time"]:
            break
        if candidate_start < entry["completion_time"]:
            candidate_start = entry["completion_time"]

    return candidate_start


def assignment_is_feasible(
    vessel: Vessel,
    berth: Berth,
    start_time: int,
    schedule: List[Dict],
) -> bool:
    """Check capacity, arrival, and non-overlap constraints for an assignment."""

    if vessel.length > berth.capacity or start_time < vessel.arrival_time:
        return False

    completion_time = start_time + vessel.processing_times[berth.id]
    for entry in schedule:
        if entry["berth_id"] != berth.id:
            continue
        if not (
            completion_time <= entry["start_time"]
            or start_time >= entry["completion_time"]
        ):
            return False
    return True


class ExplainableAISystem:
    """Rank feasible assignments using a weighted, inspectable heuristic."""

    def __init__(self, vessels: List[Vessel], berths: List[Berth]):
        self.vessels = vessels
        self.berths = berths
        self.feature_weights = {
            "vessel_priority": 0.25,
            "customer_importance": 0.20,
            "berth_productivity": 0.15,
            "schedule_flexibility": 0.10,
            "historical_performance": 0.10,
            "berth_preference_match": 0.10,
            "processing_time_efficiency": 0.10,
        }

    def calculate_feature_importance(
        self, vessel: Vessel, berth: Berth, start_time: int
    ) -> Dict[str, float]:
        """Return weighted feature contributions for one candidate assignment."""

        priority_score = {"VIP": 1.0, "High": 0.75, "Normal": 0.5, "Low": 0.25}[vessel.priority]
        processing_time = vessel.processing_times[berth.id]
        preference_score = berth.location_preference.get(vessel.priority, 0.5)

        return {
            "vessel_priority": priority_score * self.feature_weights["vessel_priority"],
            "customer_importance": (vessel.customer_importance / 5.0)
            * self.feature_weights["customer_importance"],
            "berth_productivity": (berth.productivity / 200.0)
            * self.feature_weights["berth_productivity"],
            "schedule_flexibility": vessel.schedule_flexibility
            * self.feature_weights["schedule_flexibility"],
            "historical_performance": vessel.historical_performance
            * self.feature_weights["historical_performance"],
            "berth_preference_match": preference_score
            * self.feature_weights["berth_preference_match"],
            "processing_time_efficiency": max(0.0, 1.0 - processing_time / 10.0)
            * self.feature_weights["processing_time_efficiency"],
        }

    def generate_decision(
        self, vessel: Vessel, current_schedule: List[Dict]
    ) -> AIDecision:
        """Score feasible berth assignments and return the best candidate."""

        start_clock = time.perf_counter()
        candidates = []

        for berth in self.berths:
            start_time = earliest_feasible_start(vessel, berth, current_schedule)
            if start_time is None:
                continue

            features = self.calculate_feature_importance(vessel, berth, start_time)
            waiting_time = start_time - vessel.arrival_time
            score = sum(features.values()) - 0.05 * waiting_time
            processing_time = vessel.processing_times[berth.id]
            candidates.append(
                {
                    "berth_id": berth.id,
                    "start_time": start_time,
                    "completion_time": start_time + processing_time,
                    "decision_score": score,
                    "features": features,
                    "waiting_time": waiting_time,
                    "processing_time": processing_time,
                }
            )

        candidates.sort(key=lambda candidate: candidate["decision_score"], reverse=True)
        elapsed_ms = (time.perf_counter() - start_clock) * 1000

        if not candidates:
            return AIDecision(
                vessel_id=vessel.id,
                recommended_berth=-1,
                recommended_start_time=-1,
                confidence_score=0.0,
                feature_importance={},
                alternative_solutions=[],
                reasoning="No feasible berth assignment is available.",
                processing_time_ms=elapsed_ms,
            )

        best = candidates[0]
        if len(candidates) > 1:
            score_margin = best["decision_score"] - candidates[1]["decision_score"]
            confidence_score = min(1.0, max(0.3, score_margin * 2))
        else:
            confidence_score = 0.7

        alternatives = [
            {
                "berth_id": candidate["berth_id"],
                "start_time": candidate["start_time"],
                "decision_score": candidate["decision_score"],
                "trade_offs": (
                    f"Lower score by "
                    f"{best['decision_score'] - candidate['decision_score']:.2f}"
                ),
            }
            for candidate in candidates[1:4]
        ]

        selected_berth = next(
            berth for berth in self.berths if berth.id == best["berth_id"]
        )
        reasoning = (
            f"Recommended Berth {best['berth_id']} at time {best['start_time']} "
            f"with heuristic score {best['decision_score']:.2f}. Key context: "
            f"{vessel.priority} priority, berth productivity "
            f"{selected_berth.productivity} TEU/hour, and waiting time "
            f"{best['waiting_time']} hours."
        )

        return AIDecision(
            vessel_id=vessel.id,
            recommended_berth=best["berth_id"],
            recommended_start_time=best["start_time"],
            confidence_score=confidence_score,
            feature_importance=best["features"],
            alternative_solutions=alternatives,
            reasoning=reasoning,
            processing_time_ms=elapsed_ms,
        )


print("Explainable AI assignment scorer is ready.")

## 3. Human-operator rules and decision modes

The operator model first computes the stated complexity score, then selects AI-led, human-led, or collaborative authority according to the thresholds above. Its rules express different decision roles; they do not model a trained human or establish that one mode is inherently superior. Every resulting assignment is still checked against berth capacity, vessel arrival, and the existing schedule.

In [ ]:
class HumanOperator:
    """Apply explicit operator rules while considering the AI recommendation."""

    def __init__(
        self,
        berths: List[Berth],
        experience_years: int = 10,
        specialization: str = "Berth Allocation",
    ):
        self.berths = berths
        self.experience_years = experience_years
        self.specialization = specialization
        self.trust_score = 0.7
        self.decision_history = []
        self.expertise_patterns = {
            "VIP_vessels": 0.95,
            "emergency_situations": 0.88,
            "capacity_constraints": 0.82,
            "schedule_optimization": 0.90,
            "customer_relations": 0.85,
        }

    def assess_situation_complexity(
        self, vessel: Vessel, ai_decision: AIDecision
    ) -> float:
        """Combine normalized contextual factors into a score from zero to one."""

        slack = vessel.desired_departure - vessel.arrival_time
        complexity_factors = {
            "vip_vessel": 1.0 if vessel.priority == "VIP" else 0.0,
            "high_importance": (
                (vessel.customer_importance - 2.0) / 3.0
                if vessel.customer_importance > 2.0
                else 0.0
            ),
            "low_confidence": max(
                0.0, (0.7 - ai_decision.confidence_score) / 0.7
            ),
            "tight_schedule": max(0.0, (8.0 - slack) / 8.0),
            "low_flexibility": max(0.0, 1.0 - vessel.schedule_flexibility),
            "performance_issues": max(
                0.0, (0.9 - vessel.historical_performance) / 0.9
            ),
        }
        weights = {
            "vip_vessel": 0.3,
            "high_importance": 0.2,
            "low_confidence": 0.2,
            "tight_schedule": 0.1,
            "low_flexibility": 0.1,
            "performance_issues": 0.1,
        }
        complexity = sum(
            complexity_factors[name] * weights[name]
            for name in complexity_factors
        )
        return min(1.0, complexity)

    def determine_decision_mode(
        self, vessel: Vessel, ai_decision: AIDecision
    ) -> DecisionMode:
        """Select authority using vessel context, complexity, and AI confidence."""

        complexity = self.assess_situation_complexity(vessel, ai_decision)
        if vessel.priority == "VIP" or complexity > 0.7:
            return DecisionMode.HUMAN_LED
        if ai_decision.confidence_score > 0.85 and complexity < 0.3:
            return DecisionMode.AI_LED
        return DecisionMode.COLLABORATIVE

    def make_human_decision(
        self,
        vessel: Vessel,
        ai_decision: AIDecision,
        current_schedule: List[Dict],
    ) -> HumanDecision:
        """Choose a feasible assignment according to the selected decision mode."""

        start_clock = time.perf_counter()
        decision_mode = self.determine_decision_mode(vessel, ai_decision)

        if decision_mode == DecisionMode.AI_LED:
            final_berth = ai_decision.recommended_berth
            final_start_time = ai_decision.recommended_start_time
            reasoning = f"Accepted the AI recommendation: {ai_decision.reasoning}"
            confidence_level = ai_decision.confidence_score * 0.9
            ai_agreement_score = 1.0

        elif decision_mode == DecisionMode.HUMAN_LED:
            preferred_berth_id = (
                1 if vessel.priority == "VIP" else (vessel.id % len(self.berths)) + 1
            )
            feasible_berths = [
                berth for berth in self.berths if vessel.length <= berth.capacity
            ]
            if not feasible_berths:
                raise ValueError(f"No berth has enough capacity for vessel {vessel.id}.")

            preferred_berth = next(
                (
                    berth
                    for berth in feasible_berths
                    if berth.id == preferred_berth_id
                ),
                feasible_berths[0],
            )
            final_berth = preferred_berth.id
            final_start_time = earliest_feasible_start(
                vessel, preferred_berth, current_schedule
            )
            if final_start_time is None:
                raise ValueError(f"No feasible start time for vessel {vessel.id}.")

            reasoning = (
                f"Human-led rule selected Berth {final_berth} for a "
                f"{vessel.priority} priority vessel and scheduled its earliest "
                f"available start time."
            )
            confidence_level = 0.85
            ai_agreement_score = (
                0.8 if final_berth == ai_decision.recommended_berth else 0.3
            )

        else:
            final_berth = ai_decision.recommended_berth
            final_start_time = ai_decision.recommended_start_time
            reasoning = "Collaborative review retained the AI recommendation."
            ai_agreement_score = 0.8

            if (
                ai_decision.confidence_score <= 0.7
                or vessel.priority == "VIP"
            ) and ai_decision.alternative_solutions:
                alternative = ai_decision.alternative_solutions[0]
                final_berth = alternative["berth_id"]
                final_start_time = alternative["start_time"]
                reasoning = (
                    "Collaborative review selected a feasible AI alternative: "
                    f"{alternative['trade_offs']}."
                )
                ai_agreement_score = 0.6
            elif (
                vessel.schedule_flexibility > 0.7
                and final_start_time > vessel.arrival_time
            ):
                delayed_start = final_start_time + 1
                selected_berth = next(
                    berth for berth in self.berths if berth.id == final_berth
                )
                if assignment_is_feasible(
                    vessel, selected_berth, delayed_start, current_schedule
                ):
                    final_start_time = delayed_start
                    reasoning = (
                        "Collaborative review delayed the feasible AI assignment "
                        "by one hour to represent a human scheduling adjustment."
                    )

            confidence_level = (ai_decision.confidence_score + 0.85) / 2.0

        selected_berth = next(
            berth for berth in self.berths if berth.id == final_berth
        )
        if not assignment_is_feasible(
            vessel, selected_berth, final_start_time, current_schedule
        ):
            # A recommendation is advisory; an infeasible assignment cannot enter the schedule.
            final_berth = ai_decision.recommended_berth
            final_start_time = ai_decision.recommended_start_time
            selected_berth = next(
                berth for berth in self.berths if berth.id == final_berth
            )
            if not assignment_is_feasible(
                vessel, selected_berth, final_start_time, current_schedule
            ):
                raise ValueError(f"No feasible final assignment for vessel {vessel.id}.")
            reasoning += " The feasible AI recommendation was retained for scheduling."
            ai_agreement_score = 1.0

        elapsed_seconds = time.perf_counter() - start_clock
        decision = HumanDecision(
            vessel_id=vessel.id,
            final_berth=final_berth,
            final_start_time=final_start_time,
            decision_mode=decision_mode,
            reasoning=reasoning,
            confidence_level=confidence_level,
            time_to_decide_seconds=elapsed_seconds,
            ai_agreement_score=ai_agreement_score,
        )
        self.decision_history.append(decision)
        return decision

    def update_trust_score(
        self, decision: HumanDecision, actual_performance: float
    ) -> None:
        """Update trust only when an externally observed outcome is available."""

        if actual_performance > 0.8:
            if decision.decision_mode == DecisionMode.AI_LED:
                self.trust_score = min(1.0, self.trust_score + 0.02)
            elif decision.ai_agreement_score > 0.7:
                self.trust_score = min(1.0, self.trust_score + 0.01)
        elif actual_performance < 0.6 and decision.decision_mode == DecisionMode.AI_LED:
            self.trust_score = max(0.3, self.trust_score - 0.03)


print("Human-operator decision rules are ready.")

## 4. Partnership manager and schedule construction

The manager processes each vessel once, in the order supplied. It records the AI recommendation and final operator decision, adds the assignment to the schedule, and updates descriptive counts and averages. The schedule validation checks that every assignment meets the modeled capacity and non-overlap constraints. No outcome-based trust update is made because this simulation does not provide observed service-quality outcomes.

In [ ]:
class HumanAIPartnershipManager:
    """Coordinate sequential recommendations, decisions, and schedule records."""

    def __init__(self, vessels: List[Vessel], berths: List[Berth]):
        self.vessels = vessels
        self.berths = berths
        self.ai_system = ExplainableAISystem(vessels, berths)
        self.human_operator = HumanOperator(berths)
        self.current_schedule = []
        self.metrics = PartnershipMetrics(trust_score=self.human_operator.trust_score)
        self.decision_log = []

    def process_vessel_arrival(self, vessel: Vessel) -> HumanDecision:
        """Process one vessel and append its feasible assignment to the schedule."""

        ai_decision = self.ai_system.generate_decision(vessel, self.current_schedule)
        human_decision = self.human_operator.make_human_decision(
            vessel, ai_decision, self.current_schedule
        )
        berth = next(
            berth for berth in self.berths if berth.id == human_decision.final_berth
        )
        if not assignment_is_feasible(
            vessel,
            berth,
            human_decision.final_start_time,
            self.current_schedule,
        ):
            raise AssertionError("The final assignment must satisfy schedule constraints.")

        processing_time = vessel.processing_times[berth.id]
        schedule_entry = {
            "vessel_id": vessel.id,
            "berth_id": berth.id,
            "start_time": human_decision.final_start_time,
            "completion_time": human_decision.final_start_time + processing_time,
            "processing_time": processing_time,
            "decision_mode": human_decision.decision_mode.value,
            "ai_confidence": ai_decision.confidence_score,
            "human_confidence": human_decision.confidence_level,
        }
        self.current_schedule.append(schedule_entry)
        self.decision_log.append(
            {
                "vessel": vessel,
                "ai_decision": ai_decision,
                "human_decision": human_decision,
                "schedule_entry": schedule_entry,
            }
        )
        self.update_metrics(human_decision)
        return human_decision

    def update_metrics(self, decision: HumanDecision) -> None:
        """Update mode counts and running descriptive averages."""

        self.metrics.total_decisions += 1
        if decision.decision_mode == DecisionMode.AI_LED:
            self.metrics.ai_led_decisions += 1
            acceptance_weight = 1.0
        elif decision.decision_mode == DecisionMode.HUMAN_LED:
            self.metrics.human_led_decisions += 1
            acceptance_weight = (
                0.8 if decision.ai_agreement_score > 0.7 else 0.3
            )
        else:
            self.metrics.collaborative_decisions += 1
            acceptance_weight = (
                0.8 if decision.ai_agreement_score > 0.7 else 0.3
            )

        count = self.metrics.total_decisions
        self.metrics.ai_acceptance_rate += (
            acceptance_weight - self.metrics.ai_acceptance_rate
        ) / count
        self.metrics.average_decision_time_seconds += (
            decision.time_to_decide_seconds
            - self.metrics.average_decision_time_seconds
        ) / count
        self.metrics.trust_score = self.human_operator.trust_score
        self.metrics.operator_satisfaction = (
            0.6 * self.metrics.trust_score
            + 0.4 * self.metrics.ai_acceptance_rate
        )

    def calculate_performance_metrics(self) -> None:
        """Calculate total waiting-plus-processing time for the completed schedule."""

        vessel_by_id = {vessel.id: vessel for vessel in self.vessels}
        self.metrics.total_service_time_hours = sum(
            entry["completion_time"] - vessel_by_id[entry["vessel_id"]].arrival_time
            for entry in self.current_schedule
        )

    def simulate_partnership_operation(self) -> Dict:
        """Process all vessels and return the schedule, metrics, and decision log."""

        print("Simulating the Human–AI partnership operation")
        print("=" * 48)
        for vessel in self.vessels:
            decision = self.process_vessel_arrival(vessel)
            print(
                f"Vessel {vessel.id} ({vessel.priority}): "
                f"{decision.decision_mode.value}, Berth {decision.final_berth}, "
                f"start {decision.final_start_time}, "
                f"computer decision time "
                f"{decision.time_to_decide_seconds:.6f} seconds"
            )

        self.calculate_performance_metrics()
        return {
            "schedule": self.current_schedule,
            "metrics": self.metrics,
            "decision_log": self.decision_log,
        }


print("Partnership manager is ready.")

In [ ]:
partnership_manager = HumanAIPartnershipManager(vessels, berths)
simulation_results = partnership_manager.simulate_partnership_operation()

metrics = simulation_results["metrics"]
total_decisions = metrics.total_decisions
assert total_decisions == len(vessels)
assert (
    metrics.ai_led_decisions
    + metrics.human_led_decisions
    + metrics.collaborative_decisions
    == total_decisions
)

print("\nPartnership simulation summary")
print("=" * 34)
print("Decision modes:")
for label, count in [
    ("AI-led", metrics.ai_led_decisions),
    ("Human-led", metrics.human_led_decisions),
    ("Collaborative", metrics.collaborative_decisions),
]:
    print(f"  {label}: {count} ({count / total_decisions:.1%})")

print("\nDescriptive metrics:")
print(f"  Mean computer decision time: {metrics.average_decision_time_seconds:.6f} seconds")
print(f"  AI acceptance proxy: {metrics.ai_acceptance_rate:.1%}")
print(f"  Initial trust score: {metrics.trust_score:.1%}")
print(f"  Composite partnership indicator: {metrics.operator_satisfaction:.1%}")
print(f"  Total waiting-plus-processing time: {metrics.total_service_time_hours:.1f} hours")
print("  Baseline improvements: not calculated; no baseline runs are included.")

schedule_df = pd.DataFrame(simulation_results["schedule"]).sort_values(
    ["start_time", "berth_id"]
)
print("\nFinal schedule (times in hours):")
print(
    schedule_df[
        ["vessel_id", "berth_id", "start_time", "completion_time", "decision_mode"]
    ].to_string(index=False)
)

# Verify that the reported schedule is feasible and its intervals match processing times.
vessel_by_id = {vessel.id: vessel for vessel in vessels}
berth_by_id = {berth.id: berth for berth in berths}
for entry in simulation_results["schedule"]:
    vessel = vessel_by_id[entry["vessel_id"]]
    berth = berth_by_id[entry["berth_id"]]
    assert vessel.length <= berth.capacity
    assert entry["start_time"] >= vessel.arrival_time
    assert entry["completion_time"] - entry["start_time"] == vessel.processing_times[berth.id]
for berth_id in berth_by_id:
    entries = sorted(
        (entry for entry in simulation_results["schedule"] if entry["berth_id"] == berth_id),
        key=lambda entry: entry["start_time"],
    )
    for first, second in zip(entries, entries[1:]):
        assert first["completion_time"] <= second["start_time"]
assert sum(
    entry["completion_time"] - vessel_by_id[entry["vessel_id"]].arrival_time
    for entry in simulation_results["schedule"]
) == metrics.total_service_time_hours
print("\nSchedule feasibility and metric consistency checks passed.")

## 5. Visualization dashboard

The first figure summarizes the decision-mode mix, proxy metrics, measured computer execution times, and berth reservations. The second figure shows recommendation confidence alongside agreement, feature contributions, the recorded schedule-time metric, and the trust value recorded by the simulation. Since trust is not updated without observed outcomes, its trace is a record of the initialized model value, not evidence of learning or adaptation. Feature contributions are averages over the recommendations actually recorded.

In [ ]:
def create_partnership_dashboard(
    results: Dict, vessels: List[Vessel], berths: List[Berth]
) -> None:
    """Plot decision, schedule, and explanation summaries from simulation results."""

    metrics = results["metrics"]
    decision_log = results["decision_log"]
    schedule = results["schedule"]
    vessel_by_id = {vessel.id: vessel for vessel in vessels}

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("Human–AI Partnership Simulation Dashboard", fontsize=15)

    modes = ["AI-Led", "Human-Led", "Collaborative"]
    counts = [
        metrics.ai_led_decisions,
        metrics.human_led_decisions,
        metrics.collaborative_decisions,
    ]
    nonzero = [(mode, count) for mode, count in zip(modes, counts) if count > 0]
    axes[0, 0].pie(
        [count for _, count in nonzero],
        labels=[mode for mode, _ in nonzero],
        autopct="%1.1f%%",
        startangle=90,
    )
    axes[0, 0].set_title("Decision mode distribution")

    quality_metrics = {
        "AI acceptance\nproxy": metrics.ai_acceptance_rate * 100,
        "Recorded\ntrust": metrics.trust_score * 100,
        "Composite\nindicator": metrics.operator_satisfaction * 100,
    }
    bars = axes[0, 1].bar(quality_metrics.keys(), quality_metrics.values())
    axes[0, 1].set_title("Partnership proxies (not survey outcomes)")
    axes[0, 1].set_ylabel("Index (%)")
    axes[0, 1].set_ylim(0, 110)
    for bar, value in zip(bars, quality_metrics.values()):
        axes[0, 1].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{value:.1f}%",
            ha="center",
        )

    mode_times = {
        mode: [
            entry["human_decision"].time_to_decide_seconds
            for entry in decision_log
            if entry["human_decision"].decision_mode.value == mode
        ]
        for mode in modes
    }
    available_modes = [mode for mode in modes if mode_times[mode]]
    if available_modes:
        axes[1, 0].boxplot(
            [mode_times[mode] for mode in available_modes],
            tick_labels=available_modes,
        )
    axes[1, 0].set_title("Computer decision execution time")
    axes[1, 0].set_ylabel("Seconds (measured runtime)")
    axes[1, 0].tick_params(axis="x", rotation=15)
    axes[1, 0].grid(True, axis="y", alpha=0.3)

    color_map = plt.get_cmap("tab10")
    for entry in schedule:
        vessel_id = entry["vessel_id"]
        axes[1, 1].barh(
            entry["berth_id"],
            entry["processing_time"],
            left=entry["start_time"],
            color=color_map((vessel_id - 1) % 10),
            edgecolor="black",
        )
        axes[1, 1].text(
            entry["start_time"] + entry["processing_time"] / 2,
            entry["berth_id"],
            f"V{vessel_id}",
            ha="center",
            va="center",
            fontsize=8,
        )
    axes[1, 1].set_title("Berth allocation schedule")
    axes[1, 1].set_xlabel("Time (hours)")
    axes[1, 1].set_ylabel("Berth ID")
    axes[1, 1].set_yticks([berth.id for berth in berths])
    axes[1, 1].grid(True, axis="x", alpha=0.3)
    fig.tight_layout()
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("Decision Explanation and Recorded Indicators", fontsize=15)

    confidence = [entry["ai_decision"].confidence_score for entry in decision_log]
    agreement = [entry["human_decision"].ai_agreement_score for entry in decision_log]
    axes[0, 0].scatter(confidence, agreement, alpha=0.75)
    axes[0, 0].plot([0, 1], [0, 1], "r--", label="Equal scores")
    axes[0, 0].set_title("AI confidence and human–AI agreement")
    axes[0, 0].set_xlabel("AI confidence score")
    axes[0, 0].set_ylabel("Agreement score")
    axes[0, 0].set_xlim(0, 1)
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    all_features = {}
    for entry in decision_log:
        for feature, importance in entry["ai_decision"].feature_importance.items():
            all_features.setdefault(feature, []).append(importance)
    feature_averages = {
        feature: float(np.mean(values)) for feature, values in all_features.items()
    }
    ordered_features = sorted(
        feature_averages.items(), key=lambda item: item[1]
    )
    axes[0, 1].barh(
        [name.replace("_", " ").title() for name, _ in ordered_features],
        [value for _, value in ordered_features],
    )
    axes[0, 1].set_title("Mean weighted feature contributions")
    axes[0, 1].set_xlabel("Contribution to heuristic score")
    axes[0, 1].grid(True, axis="x", alpha=0.3)

    vessel_ids = [entry["vessel_id"] for entry in schedule]
    waiting_times = [
        entry["start_time"] - vessel_by_id[entry["vessel_id"]].arrival_time
        for entry in schedule
    ]
    axes[1, 0].bar(vessel_ids, waiting_times)
    axes[1, 0].set_title("Waiting time by vessel")
    axes[1, 0].set_xlabel("Vessel ID")
    axes[1, 0].set_ylabel("Waiting time (hours)")
    axes[1, 0].grid(True, axis="y", alpha=0.3)

    recorded_trust = [
        entry["human_decision"].confidence_level * 0
        + metrics.trust_score
        for entry in decision_log
    ]
    axes[1, 1].plot(
        range(1, len(recorded_trust) + 1),
        recorded_trust,
        marker="o",
    )
    axes[1, 1].set_title("Recorded trust (no outcome-based updates)")
    axes[1, 1].set_xlabel("Decision number")
    axes[1, 1].set_ylabel("Trust score")
    axes[1, 1].set_ylim(0, 1)
    axes[1, 1].grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


create_partnership_dashboard(simulation_results, vessels, berths)

## 6. Inspecting one explainable decision

The following example examines the third vessel in the recorded decision log. It prints the recommendation, the weighted feature contributions, any ranked alternatives, the operator's final decision, and the complexity score. The listed contributions explain the AI's heuristic score; they are not causal effects or statistical feature-importance estimates.

In [ ]:
def demonstrate_explainable_ai(
    results: Dict, manager: HumanAIPartnershipManager
) -> None:
    """Print the recommendation and operator decision for one recorded vessel."""

    if len(results["decision_log"]) < 3:
        print("At least three recorded decisions are needed for this example.")
        return

    sample = results["decision_log"][2]
    vessel = sample["vessel"]
    ai_decision = sample["ai_decision"]
    human_decision = sample["human_decision"]

    print("Explainable AI decision inspection")
    print("=" * 38)
    print(f"Vessel {vessel.id} ({vessel.priority} priority)")
    print(f"AI berth recommendation: {ai_decision.recommended_berth}")
    print(f"AI start-time recommendation: {ai_decision.recommended_start_time} hours")
    print(f"AI confidence margin score: {ai_decision.confidence_score:.3f}")
    print(f"Computer evaluation time: {ai_decision.processing_time_ms:.6f} ms")
    print(f"AI rationale: {ai_decision.reasoning}")

    print("\nWeighted feature contributions:")
    for feature, value in sorted(
        ai_decision.feature_importance.items(),
        key=lambda item: item[1],
        reverse=True,
    ):
        print(f"  {feature.replace('_', ' ').title():32s} {value:.3f}")

    print("\nFeasible ranked alternatives:")
    if not ai_decision.alternative_solutions:
        print("  No alternative berth candidate was available.")
    for index, alternative in enumerate(ai_decision.alternative_solutions, start=1):
        print(
            f"  {index}. Berth {alternative['berth_id']} at "
            f"{alternative['start_time']} hours; "
            f"score {alternative['decision_score']:.3f}; "
            f"{alternative['trade_offs']}"
        )

    print("\nOperator decision:")
    print(
        f"  Berth {human_decision.final_berth} at "
        f"{human_decision.final_start_time} hours"
    )
    print(f"  Mode: {human_decision.decision_mode.value}")
    print(f"  Agreement proxy: {human_decision.ai_agreement_score:.2f}")
    print(f"  Operator-model confidence: {human_decision.confidence_level:.2f}")
    print(f"  Rule-based rationale: {human_decision.reasoning}")

    complexity = manager.human_operator.assess_situation_complexity(
        vessel, ai_decision
    )
    print(f"  Situation complexity score: {complexity:.3f} on a 0–1 scale")


demonstrate_explainable_ai(simulation_results, partnership_manager)

## 7. Interpreting the results and evaluation limits

The schedule and decision log describe the behavior of the specified sequential rules on the fixed instance. The mode distribution is determined by the coded thresholds; the feature contributions are determined by the specified weights; and the recorded waiting and service times are calculated from the resulting schedule.

The AI acceptance proxy is a running average of rule-assigned agreement weights. The composite partnership indicator combines that proxy with the initialized trust score. Neither quantity is a measured acceptance rate or operator-satisfaction survey result. Computer execution time is dependent on the environment and is not an estimate of human decision latency.

The notebook does not include independently run AI-only or human-only baselines, observed operational outcomes, disruption events, statistical confidence intervals, or a global optimality benchmark. Consequently, it does not report comparative performance improvements, learning over time, or real-world effectiveness. Extending the instance with operational observations or additional constraints would require defining and validating those inputs; this notebook's results alone do not establish those claims.

## 8. Key takeaways

- The partnership is implemented as a sequence of transparent rules: score feasible AI candidates, select a decision mode, produce a human–AI assignment, and validate the schedule.
- The score and confidence value are heuristic quantities whose meaning follows from their definitions; neither is a calibrated probability or a proof of optimality.
- Capacity, arrival-time, and berth non-overlap checks make the generated schedule feasible under this model. Input order and local candidate ranking mean that feasibility does not imply global optimality.
- A plotted or printed metric should be interpreted according to how it is computed. Proxy indices and computer runtime do not substitute for measured operator satisfaction, real response times, or baseline experiments.
- Human oversight is represented here by explicit decision rules. Actual deployment would additionally require validated operational data, accountable review procedures, and evaluation against appropriate observed outcomes.